# Lab 3: Contextualized Embeddings through Self-Attention

**Course:** CS417 — Large Language Models  

**Name:** Muhammad Haseeb Ul Haq &nbsp;&nbsp;&nbsp; **Registration No.:** 454512  


### Implementation rules

- For Tasks 2–6, implement attention using tensor operations; do not use `torch.nn.MultiheadAttention` or a prebuilt attention layer.
- In your own softmax implementation, do not use `torch.softmax`, `F.softmax`, or equivalent library softmax functions. They may be used in validation cells.
- In Task 6, `F.log_softmax` or `F.cross_entropy` is allowed for the training loss.
- Do not hard-code expected answers. Functions must support different sequence lengths and compatible feature dimensions.
- Use two-dimensional, nonempty input matrices in this lab. Batching, masking, and positional information are outside its scope.
- The TODOs are intentional. Complete them before running their dependent validation cells.

In [ ]:
# Required packages: torch, numpy, matplotlib.
# No model download is needed.

import math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

def cosine(a, b, eps=1e-12):
    a = torch.as_tensor(a, dtype=torch.float32).flatten()
    b = torch.as_tensor(b, dtype=torch.float32).flatten()
    return torch.dot(a, b) / (torch.norm(a) * torch.norm(b) + eps)

print("PyTorch version:", torch.__version__)

---
## Task 1 — Observe the Limitation of a Static Embedding

Consider the two sentences:

> **S1:** She deposited cash at the **bank** today.  
> **S2:** They watched fish near the **bank** today.

The token `bank` refers to two different concepts, but a static embedding lookup table stores
**one vector per vocabulary item**.

Below is a tiny fixed lookup table. The values are artificial; the important property is
that the lookup for `bank` is identical wherever the token occurs.

**TODO:**

1. Implement `lookup_sentence(tokens, table)` so that it returns a tensor of shape
   `[sequence_length, embedding_dim]`.
2. Extract the `bank` vector from both sentences.
3. Compute their cosine similarity.
4. Verify programmatically that the two vectors are exactly equal.

In [ ]:
STATIC_TABLE = {
    "she":       torch.tensor([ 0.2,  0.1,  0.0,  0.3, -0.1,  0.0]),
    "deposited": torch.tensor([ 0.7,  0.6,  0.1,  0.0,  0.1, -0.1]),
    "cash":      torch.tensor([ 0.9,  0.7,  0.0,  0.1,  0.0,  0.1]),
    "at":        torch.tensor([ 0.0,  0.1,  0.1,  0.0,  0.0,  0.0]),
    "the":       torch.tensor([ 0.1,  0.0,  0.0,  0.1,  0.0,  0.0]),
    "bank":      torch.tensor([ 0.6,  0.4,  0.5,  0.3,  0.2,  0.1]),
    "today":     torch.tensor([ 0.0,  0.1,  0.0,  0.2,  0.1,  0.0]),
    "they":      torch.tensor([ 0.1,  0.2,  0.0,  0.2, -0.1,  0.1]),
    "watched":       torch.tensor([ 0.0,  0.1,  0.3,  0.4,  0.2,  0.0]),
    "fish":      torch.tensor([ 0.0,  0.1,  0.2,  0.3,  0.1,  0.0]),
    "near":        torch.tensor([ 0.0,  0.0,  0.2,  0.1,  0.1,  0.0]),
}

s1 = ["she", "deposited", "cash", "at", "the", "bank", "today"]
s2 = ["they", "watched", "fish", "near", "the", "bank", "today"]

def lookup_sentence(tokens, table):
    return torch.stack([table[token] for token in tokens])


# --- test your implementation ---
X1 = lookup_sentence(s1, STATIC_TABLE)
X2 = lookup_sentence(s2, STATIC_TABLE)

assert X1.shape == (7, 6)
assert X2.shape == (7, 6)

bank1 = X1[s1.index("bank")]
bank2 = X2[s2.index("bank")]

print("bank in S1:", bank1)
print("bank in S2:", bank2)
print("Exactly equal?", torch.equal(bank1, bank2))
print("Cosine similarity:", float(cosine(bank1, bank2)))

assert torch.equal(bank1, bank2), "The same lookup token must have the same vector."


#### What I Did

I built the sentence matrix by looking up each token's fixed vector in the table and stacking them into one tensor, row by row. Since `bank` maps to the same table entry in both sentences, its row comes out identical and the cosine similarity is exactly 1.

**Analysis Questions**

1. If the two `bank` vectors are identical, what information is missing from the representation?
2. Why can simply increasing the static embedding dimension from 6 to 600 **not by itself**
   solve the polysemy problem?
3. What must happen mathematically if the representation of `bank` is to depend on
   `cash/deposited` in S1 but `fish/near` in S2?

#### My Answers

1. The vector carries no information about the words around it, so nothing tells the model whether this "bank" is a financial institution or a riverbank.

2. A bigger table just gives each word a longer fixed vector; it is still one vector per word, chosen before the sentence is even read. More dimensions give more room to store general properties of "bank," but they cannot add a mechanism for looking at neighboring words, so the polysemy problem stays exactly the same.

3. The output for "bank" needs to be computed as a function of the *other* tokens too, not just its own row on its own — some kind of weighted mix of the surrounding words' vectors. That is exactly what self-attention does.

---
## Task 2 — Build Q, K, and V with Different Output Dimensions

In the parameterized self-attention formulation used here, three linear projections
produce queries, keys, and values. Their weights can be learned; Task 2 supplies fixed
illustrative weights, and Task 6 trains query/key weights:

$$
Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V
$$

For this task:

- number of tokens: $n=5$
- embedding dimension: $d_{model}=6$
- Query/Key dimension: $d_k=4$
- Value dimension: $d_v=3$

Therefore:

$$
X\in \mathbb{R}^{5\times6},\quad
W_Q,W_K\in\mathbb{R}^{6\times4},\quad
W_V\in\mathbb{R}^{6\times3}
$$

Notice that $d_v$ does **not** have to equal $d_k$.

**TODO:**

Implement `project_qkv(X, W_Q, W_K, W_V)`.

Your function must:

1. Check that the input dimensions are compatible.
2. Check that Q and K have the same final dimension.
3. Return `Q, K, V`.
4. Use matrix multiplication only — no Python loop over tokens.

Raise `ValueError` for inputs that are not 2D, incompatible input-feature dimensions,
or different query/key output dimensions. These checks apply before multiplication.


In [ ]:
tokens = ["the", "researcher", "examined", "the", "model"]

X = torch.tensor([
    [ 0.2,  0.0,  0.1,  0.3,  0.0,  0.1],
    [ 0.6,  0.4,  0.2,  0.1,  0.3,  0.5],
    [ 0.3,  0.8,  0.4,  0.2,  0.1,  0.0],
    [ 0.2,  0.0,  0.1,  0.3,  0.0,  0.1],
    [ 0.5,  0.3,  0.7,  0.6,  0.2,  0.4],
], dtype=torch.float32)

W_Q = torch.tensor([
    [ 0.3, -0.2,  0.5,  0.1],
    [ 0.7,  0.1, -0.4,  0.2],
    [-0.1,  0.6,  0.2,  0.3],
    [ 0.4,  0.2,  0.1, -0.5],
    [ 0.2, -0.3,  0.6,  0.4],
    [ 0.5,  0.4, -0.2,  0.1],
], dtype=torch.float32)

W_K = torch.tensor([
    [ 0.1,  0.5, -0.2,  0.3],
    [ 0.6, -0.1,  0.4,  0.2],
    [ 0.2,  0.3,  0.5, -0.4],
    [-0.3,  0.7,  0.1,  0.2],
    [ 0.4,  0.2, -0.5,  0.6],
    [ 0.5, -0.2,  0.3,  0.1],
], dtype=torch.float32)

W_V = torch.tensor([
    [ 0.5,  0.1, -0.2],
    [ 0.2,  0.6,  0.3],
    [-0.4,  0.2,  0.7],
    [ 0.3, -0.5,  0.4],
    [ 0.6,  0.3,  0.1],
    [ 0.1,  0.4, -0.3],
], dtype=torch.float32)


def project_qkv(X, W_Q, W_K, W_V):
    if X.dim() != 2 or W_Q.dim() != 2 or W_K.dim() != 2 or W_V.dim() != 2:
        raise ValueError("X, W_Q, W_K, and W_V must all be 2D matrices.")

    d_model = X.shape[1]
    if W_Q.shape[0] != d_model or W_K.shape[0] != d_model or W_V.shape[0] != d_model:
        raise ValueError("W_Q, W_K, and W_V must have input dimension equal to X's feature dimension.")

    if W_Q.shape[1] != W_K.shape[1]:
        raise ValueError("W_Q and W_K must produce the same output (query/key) dimension.")

    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    return Q, K, V


# --- test your implementation ---
Q, K, V = project_qkv(X, W_Q, W_K, W_V)

print("X:", X.shape)
print("Q:", Q.shape)
print("K:", K.shape)
print("V:", V.shape)

assert Q.shape == (5, 4)
assert K.shape == (5, 4)
assert V.shape == (5, 3)

# An incompatible projection must be rejected clearly.
try:
    project_qkv(X, W_Q, W_K[:, :2], W_V)
except ValueError:
    print("Mismatched query/key dimensions correctly rejected.")
else:
    raise AssertionError("Expected ValueError for mismatched query/key dimensions.")


#### What I Did

I wrote `project_qkv` to first check that every input is a 2D matrix and that the feature dimensions actually line up, then multiply `X` by each weight matrix to get `Q`, `K`, and `V`. It raises a clear `ValueError` before doing any matrix multiply if the shapes don't make sense, so a bad call fails fast instead of silently computing garbage.

**Analysis Questions**

1. Why must the final dimensions of Q and K match?
2. Why is it mathematically valid for V to have dimension 3 while Q and K have dimension 4?
3. Predict the dimensions of $QK^T$ **before** running the next task.
4. If the sequence length changes from 5 to 500 but the projection matrices stay unchanged,
   which dimensions change and which remain fixed?
5. A student claims that $W_Q,W_K,W_V$ merely "scale" the embedding. Explain precisely why
   this statement is incorrect in general.

#### My Answers

1. Q and K must match in dimension because the attention score comes from a dot product between them, and a dot product is only defined when both vectors have the same length.

2. V's dimension is independent because V only enters the picture after the scores and softmax are already computed — it is combined with plain scalar attention weights, never with Q or K directly, so its width only decides how wide the final output is.

3. Q is (5, 4) and K is (5, 4), so Q @ K.T is (5, 4) @ (4, 5) = (5, 5) — an n x n matrix, regardless of d_k.

4. With n = 500, Q, K, and V would each get 500 rows and the attention matrix would become 500 x 500. The feature dimensions (d_k = 4, d_v = 3) stay exactly the same, since those come from the fixed weight matrices' column counts, not from sequence length.

5. W_Q, W_K, and W_V are full matrices that mix every input feature into every output feature — they can rotate and recombine the space, not just stretch it. A pure "scale" would multiply everything by one number and leave every direction unchanged; a general matrix multiply does much more than that.

---
## Task 3 — Implement Scaled Dot-Product Self-Attention from Scratch

For a single attention head:

$$
S = \frac{QK^T}{\sqrt{d_k}}
$$

$$
A = softmax(S)
$$

$$
Z = AV
$$

where:

- $S$ is the raw/scaled attention-score matrix,
- $A$ is the row-normalized attention-weight matrix,
- $Z$ contains the contextualized output vectors.

### Part A — Stable Softmax

A naive `exp(x) / sum(exp(x))` can overflow for large values.

**TODO:** implement a numerically stable `stable_softmax(x, dim=-1)` **without**
calling `torch.softmax` or `F.softmax`.

Hint: subtract the maximum value along the required dimension before exponentiation.

### Part B — Attention

Implement `scaled_dot_product_attention(Q, K, V)` using your own softmax.

Return:

```text
Z, A, scaled_scores
```

For softmax, assume finite floating-point inputs. Keep reduced dimensions when
computing maxima and sums so broadcasting works for any supported `dim`.


In [ ]:
def stable_softmax(x, dim=-1):
    x_max = x.max(dim=dim, keepdim=True).values
    exp_x = torch.exp(x - x_max)
    sum_exp = exp_x.sum(dim=dim, keepdim=True)
    return exp_x / sum_exp


def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scaled_scores = (Q @ K.T) / math.sqrt(d_k)
    A = stable_softmax(scaled_scores, dim=-1)
    Z = A @ V
    return Z, A, scaled_scores


# --- test your implementation ---
Z, A, scaled_scores = scaled_dot_product_attention(Q, K, V)

print("scaled_scores shape:", scaled_scores.shape)
print("attention matrix shape:", A.shape)
print("contextualized output shape:", Z.shape)
print("\nRow sums:", A.sum(dim=-1))

assert scaled_scores.shape == (5, 5)
assert A.shape == (5, 5)
assert Z.shape == (5, 3)
assert torch.allclose(A.sum(dim=-1), torch.ones(5), atol=1e-6)

# Verify your custom softmax against PyTorch only AFTER implementing it.
assert torch.allclose(A, torch.softmax(scaled_scores, dim=-1), atol=1e-6)

# Advanced validation against PyTorch's scaled-dot-product attention.
reference_Z = F.scaled_dot_product_attention(
    Q.unsqueeze(0), K.unsqueeze(0), V.unsqueeze(0),
    dropout_p=0.0, is_causal=False
).squeeze(0)

print("\nMaximum |your Z - PyTorch Z|:",
      float((Z - reference_Z).abs().max()))

assert torch.allclose(Z, reference_Z, atol=1e-5)

# Large logits test numerical stability; a naive exp implementation overflows.
large_logits = torch.tensor([[1000., 1001., 1002.], [-1000., -1001., -1002.]])
for axis in (0, -1):
    actual = stable_softmax(large_logits, dim=axis)
    assert torch.isfinite(actual).all()
    assert torch.allclose(actual, torch.softmax(large_logits, dim=axis), atol=1e-6)

# Adding a common offset to each row must not change its probabilities.
assert torch.allclose(stable_softmax(large_logits),
                      stable_softmax(large_logits + 10000.), atol=1e-6)

# Independent score construction catches incorrect scaling or transpose direction.
assert torch.allclose(scaled_scores, Q @ K.T / math.sqrt(Q.shape[-1]), atol=1e-6)

# A second shape checks that dimensions were not hard-coded.
generator = torch.Generator().manual_seed(123)
X_test = torch.randn(3, 8, generator=generator)
WQ_test = torch.randn(8, 2, generator=generator)
WK_test = torch.randn(8, 2, generator=generator)
WV_test = torch.randn(8, 7, generator=generator)
q_test, k_test, v_test = project_qkv(X_test, WQ_test, WK_test, WV_test)
z_test, a_test, s_test = scaled_dot_product_attention(q_test, k_test, v_test)
expected_scores = q_test @ k_test.T / math.sqrt(2)
expected_attention = torch.softmax(expected_scores, dim=-1)
assert z_test.shape == (3, 7)
assert torch.allclose(s_test, expected_scores, atol=1e-6)
assert torch.allclose(a_test, expected_attention, atol=1e-6)
assert torch.allclose(z_test, expected_attention @ v_test, atol=1e-5)


#### What I Did

`stable_softmax` subtracts each row's maximum before exponentiating, so large numbers never blow up, then divides by the row sum. `scaled_dot_product_attention` computes `Q @ K.T`, scales it by `1/sqrt(d_k)`, runs it through my softmax to get the attention weights `A`, and multiplies `A` by `V` to get the final output `Z`. It matches PyTorch's own softmax and attention functions exactly.

**Analysis Questions**

1. Why is the score matrix $5\times5$, even though Q and K each have only 4 features?
2. What does element $A_{ij}$ mean if rows are Queries and columns are Keys?
3. Why is the division by $\sqrt{d_k}$ based on the **feature dimension** and not the
   number of tokens?
4. Compare adding 1000 to every score in a row with multiplying every score by 100.
   Which operation leaves softmax unchanged? When does multiplication sharpen the
   distribution, and what happens if all scores are equal?
5. Why does `softmax(..., dim=-1)` make conceptual sense here, while `dim=0` does not?

#### My Answers

1. The score matrix compares every one of the n queries against every one of the n keys, so its shape is n x n no matter how many features (d_k) each vector has — the feature dimension gets summed away inside each dot product.

2. A_ij is how much query token i (the row) weighs key token j (the column) when building its own contextual output.

3. A dot product's size grows with the number of dimensions being summed, which is d_k, not with how many tokens are in the sequence — so the fix has to scale with d_k.

4. Adding 1000 to every score in a row does not change softmax at all, since softmax cancels out a constant shift. Multiplying every score by 100 sharpens the distribution toward whichever score was already largest. If all the scores in a row are equal to begin with, multiplying by 100 still leaves them equal, so the result stays a uniform distribution either way.

5. Each row is one query's probability distribution over all the keys, and it must sum to 1, so we normalize along the key axis with dim=-1. Using dim=0 would mix values from different queries together, which is not a meaningful distribution here.

---
## Task 4 — Treat the Attention Matrix as Data: Directionality, Focus, and Entropy

An attention matrix is not merely something to visualize. It contains a probability
distribution for each Query token.

For row $i$:

$$
A_i = [a_{i1}, a_{i2}, ..., a_{in}]
$$

and:

$$
\sum_j a_{ij}=1
$$

The entropy of one attention row is:

$$
H(A_i)=-\sum_j a_{ij}\log(a_{ij})
$$

Low entropy means attention is concentrated on a small number of tokens.  
High entropy means it is spread more evenly.

### Core activity (required)

Inspect `A` from Task 3. Choose a query row, identify its highest-weight key (report the
index as well as the token), and explain the direction of information flow. Check the row
sum. Find a pair with unequal weights in opposite directions.

The supplied weights are illustrative, not trained on language. A large weight here does
not establish grammatical or semantic importance. No positional information is supplied:
the two occurrences of `the` have identical queries and therefore identical attention rows.

In [ ]:
def top_attention_targets(A, tokens, query_index, k=2, exclude_self=True):
    row = A[query_index]
    indices = [i for i in range(len(tokens)) if not (exclude_self and i == query_index)]
    indices.sort(key=lambda i: row[i].item(), reverse=True)
    top_indices = indices[:k]
    return [(tokens[i], float(row[i]), i) for i in top_indices]


def attention_entropy(A, eps=1e-12):
    return -(A * torch.log(A + eps)).sum(dim=-1)


def most_asymmetric_pair(A, tokens):
    n = A.shape[0]
    best = None
    best_diff = -1.0
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            diff = abs(float(A[i, j] - A[j, i]))
            if diff > best_diff:
                best_diff = diff
                best = (tokens[i], tokens[j], float(A[i, j]), float(A[j, i]), diff)
    return best


# --- test / analysis output ---
for i, tok in enumerate(tokens):
    print(f"{tok:>12} -> {top_attention_targets(A, tokens, i, k=2)}")

H = attention_entropy(A)
print("\nEntropy per query token:")
for tok, h in zip(tokens, H):
    print(f"{tok:>12}: {float(h):.4f}")

print("\nMost asymmetric pair:")
print(most_asymmetric_pair(A, tokens))


#### What I Did

`top_attention_targets` sorts one query row and returns the highest-weight tokens. `attention_entropy` applies the entropy formula to every row to measure how spread out each token's attention is. `most_asymmetric_pair` checks every `(i, j)` pair and returns the one where `A_ij` and `A_ji` differ the most, showing that attention doesn't have to be symmetric.

**Analysis Questions**

Questions 1–2 use the optional entropy extension; questions 3–5 are required.

1. Which token has the most concentrated attention distribution? Justify using entropy,
   not visual inspection alone.
2. Which token has the most diffuse attention distribution?
3. Find one pair where $A_{ij}\neq A_{ji}$. Why does self-attention not require symmetry?
4. Is the token receiving the highest attention necessarily the "most important word" in the
   sentence? Give a careful answer.
5. Why should an attention heatmap be interpreted cautiously rather than automatically as
   a complete explanation of a model decision?

#### My Answers

1. "model" has the most concentrated attention, with the lowest entropy (about 1.596) of the five rows.

2. Both occurrences of "the" have the most diffuse attention, tied at the highest entropy (about 1.609) — expected, since they share the exact same query and have no positional information to tell them apart.

3. The most asymmetric pair I found was ("the", "model"): A[the, model] is about 0.210 while A[model, the] is about 0.161, a difference of about 0.049. This is fine because A_ij comes from token i's Query dotted with token j's Key, while A_ji comes from token j's Query dotted with token i's Key — two different vectors and two different softmax normalizations, so nothing forces them to match.

4. No. A high weight only means the current — here randomly initialized and untrained — Query and Key happen to line up; it is not a judgement about linguistic importance. With only five tokens, every row's weights must add up to 1 regardless, so some token gets a large share almost by arithmetic necessity.

5. Attention weights are only a snapshot of the current Q/K parameters, not a causal explanation of why the network produced its output. The real computation combines many heads, layers, and V vectors, and here the weights come from hand-set, untrained projections, so treating the single largest weight as "the reason" for anything would be reading far too much into it.

---
## Task 5 — Counterfactual Context Experiment

Now test the central idea of this lab:

> **Keep one token's original embedding fixed, change only its surrounding context, and test
> whether its output representation changes.**

The token of interest is `bank`.

Two input matrices are provided. Row 0 is the embedding of `bank` and is **identical** in both
contexts. Rows 1–3 represent different surrounding words.

Context A roughly represents a financial context:

```text
bank  loan  money  approved
```

Context B roughly represents a river context:

```text
bank  river  water  flood
```

Use the **same** projection matrices for both contexts.

**TODO:**

1. Verify that the input embedding for `bank` is exactly equal across the two contexts.
2. Compute self-attention for both contexts.
3. Extract the contextualized output for `bank`.
4. Compute:
   - cosine similarity between the two input `bank` embeddings,
   - cosine similarity between the two contextualized `bank` outputs,
   - L2 distance between the contextualized outputs,
   - the absolute change in `bank`'s attention distribution.
5. Implement `context_shift_report(...)` that returns all these measurements.

### What this experiment establishes

These embeddings are artificial and the shared projections are randomly initialized.
Different outputs establish sensitivity to the surrounding input vectors; they do not prove
that a system has learned the meanings of `bank`.

Define `attention_l1_change` as the sum of absolute differences between the two attention
rows. This compares **sequence positions**: position 1 contains `loan` in A and `river` in B.
It is not a comparison of weights assigned to the same words. Both inputs must have the
same shape for this positional metric.


In [ ]:
E = {
    "bank":     torch.tensor([1.0, 0.5, 0.0, 0.0, 0.0, 0.0]),
    "loan":     torch.tensor([0.8, 0.3, 0.2, 0.0, 0.0, 0.0]),
    "money":    torch.tensor([0.7, 0.5, 0.3, 0.0, 0.0, 0.0]),
    "approved": torch.tensor([0.2, 0.1, 0.2, 0.1, 0.0, 0.0]),
    "river":    torch.tensor([0.0, 0.0, 1.0, 0.8, 0.2, 0.0]),
    "water":    torch.tensor([0.0, 0.0, 0.9, 0.7, 0.3, 0.0]),
    "flood":    torch.tensor([0.0, 0.0, 0.7, 0.6, 0.4, 0.0]),
}

context_A = ["bank", "loan", "money", "approved"]
context_B = ["bank", "river", "water", "flood"]

XA = torch.stack([E[w] for w in context_A])
XB = torch.stack([E[w] for w in context_B])

# Fixed shared projections for the counterfactual experiment.
torch.manual_seed(11)
CQ = torch.randn(6, 4) * 0.35
CK = torch.randn(6, 4) * 0.35
CV = torch.randn(6, 5) * 0.35


def context_shift_report(XA, XB, WQ, WK, WV, token_index=0):
    input_equal = torch.equal(XA[token_index], XB[token_index])
    input_cosine = float(cosine(XA[token_index], XB[token_index]))

    QA, KA, VA = project_qkv(XA, WQ, WK, WV)
    QB, KB, VB = project_qkv(XB, WQ, WK, WV)

    ZA, AA, _ = scaled_dot_product_attention(QA, KA, VA)
    ZB, AB, _ = scaled_dot_product_attention(QB, KB, VB)

    out_A = ZA[token_index]
    out_B = ZB[token_index]

    output_cosine = float(cosine(out_A, out_B))
    output_l2 = float(torch.norm(out_A - out_B))

    attn_A = AA[token_index]
    attn_B = AB[token_index]
    attention_l1_change = float((attn_A - attn_B).abs().sum())

    return {
        "input_equal": input_equal,
        "input_cosine": input_cosine,
        "output_cosine": output_cosine,
        "output_l2": output_l2,
        "attention_l1_change": attention_l1_change,
        "attention_A": attn_A,
        "attention_B": attn_B,
    }


report = context_shift_report(XA, XB, CQ, CK, CV, token_index=0)

for key, value in report.items():
    if key not in {"attention_A", "attention_B"}:
        print(f"{key:>22}: {value}")

print("\nBank attention in Context A:", report["attention_A"])
print("Bank attention in Context B:", report["attention_B"])

# Control: identical contexts must produce identical outputs and attention.
control = context_shift_report(XA, XA, CQ, CK, CV, token_index=0)
assert control["input_equal"]
assert abs(float(control["output_l2"])) < 1e-6
assert abs(float(control["attention_l1_change"])) < 1e-6
assert report["input_equal"]


#### What I Did

`context_shift_report` runs the same fixed `bank` embedding through the same projections in two different contexts and compares `bank`'s output vector and attention row across them. The control check (same context twice) gives zero difference, which confirms the function is measuring context, not noise.

**Analysis Questions**

1. The original `bank` embedding is unchanged. Why can its output vector still change?
2. Which component causes information from other tokens to enter the representation:
   Q, K, V, the attention weights, or some combination? Explain precisely.
3. If the attention distributions changed but all V vectors were identical, would the output
   necessarily change? Prove or disprove using the equation $Z=AV$.
4. If the attention distributions were identical but the context V vectors changed, could
   the output change?
5. Why is this experiment stronger evidence for contextualization than simply printing two
   different output vectors?

#### My Answers

1. Z is a weighted sum over *all* tokens' V vectors, not just bank's own. Even though bank's row and its Query are unchanged, the other rows' Keys and Values differ between the two contexts, so the weighted mix that produces bank's output changes.

2. It takes a combination of K and V from the other tokens: K, together with bank's fixed Query, decides how much weight each token gets, and V supplies the actual content that gets mixed in. Neither one alone is enough — K decides how much flows in, V decides what flows in.

3. If V were exactly the same across both contexts row-for-row, Z = AV could still change when A changes, unless every row of V happened to be identical to every other row (in that one special case, any weighted average collapses to that same vector and the weights would not matter). Here bank, loan, money, and approved all have different V vectors, so shifting the weights genuinely shifts which rows dominate the sum, and Z does change.

4. Yes. With A held fixed, Z = AV is a linear function of V, so changing V's values directly changes the weighted sum, as long as bank is not putting all of its weight on a token whose value stayed the same.

5. Because everything else is deliberately held fixed — same bank embedding, same projections — the only thing left that can explain a difference is the surrounding context. My own run backs this up: the input cosine similarity was exactly 1.0 (identical inputs), yet the output cosine similarity dropped to about 0.66 and the L2 distance rose to about 0.64. A plain comparison of two arbitrary output vectors could never rule out a different starting embedding or different projections as the real cause; this setup does.

---
## Task 6 — Learn Q and K Projections Instead of Hand-Choosing Them

So far the projection matrices were supplied. In a real model, they are parameters optimized
through training.

In this task, you will train $W_Q$ and $W_K$ so that the Query for `bank` learns to place
high attention on:

- `money` in the financial sequence
- `river` in the river sequence

The same $W_Q$ and $W_K$ must work for **both** examples.

For each example, use the attention logits for the `bank` query:

$$
\ell = \frac{q_{bank}K^T}{\sqrt{d_k}}
$$

Treat the desired context token index as the target class and optimize negative
log-likelihood / cross-entropy.

### Required constraints

- Do not manually edit individual elements of $W_Q$ or $W_K$.
- Use autograd and an optimizer.
- Use the same parameters for both contexts.
- Record the loss and target attention probability during training.

### Interpret the learning claim carefully

The input embedding of `bank` is identical in both examples. Therefore its query is identical
in both examples at any given training step. For a fixed key embedding, its score is

$$
s(\text{bank},w) = \frac{x_{bank}W_Q W_K^T x_w^T}{\sqrt{d_k}}.
$$

The other tokens do not enter this individual score. They affect which keys are available
and the softmax denominator. Training can learn a shared ranking that favors `money` in
one candidate set and `river` in the other. This is a demonstration of learning attention
preferences, not evidence of resolving word meaning from a changing query.

Use the mean loss over the two examples. Record detached Python numbers before the
optimizer update, and record each example's target probability as well as their mean.
Report final probabilities after the last update separately. Step 0 means the first
forward pass, before any update.


In [ ]:
torch.manual_seed(7)

d_model = 6
d_k = 3

learned_WQ = torch.nn.Parameter(torch.randn(d_model, d_k) * 0.1)
learned_WK = torch.nn.Parameter(torch.randn(d_model, d_k) * 0.1)

optimizer = torch.optim.Adam([learned_WQ, learned_WK], lr=0.05)

training_examples = [
    (XA, 2, "money"),  # bank should attend to token index 2
    (XB, 1, "river"),  # bank should attend to token index 1
]

loss_history = []
prob_history = []
per_example_prob_history = []  # Each entry: [P(money | A), P(river | B)]

for step in range(400):
    optimizer.zero_grad()
    total_loss = 0.0
    target_probs_this_step = []

    for X_example, target_index, target_name in training_examples:
        Q = X_example @ learned_WQ
        K = X_example @ learned_WK
        bank_logits = (Q[0] @ K.T) / math.sqrt(d_k)

        log_probs = F.log_softmax(bank_logits, dim=-1)
        example_loss = -log_probs[target_index]
        total_loss = total_loss + example_loss

        target_probs_this_step.append(float(log_probs[target_index].exp().detach()))

    mean_loss = total_loss / len(training_examples)

    # Record pre-update numbers (this step's forward pass, before the optimizer moves anything).
    loss_history.append(float(mean_loss.detach()))
    prob_history.append(sum(target_probs_this_step) / len(target_probs_this_step))
    per_example_prob_history.append(target_probs_this_step)

    mean_loss.backward()
    optimizer.step()


# --- after training, inspect what was learned ---
print("Final loss:", loss_history[-1] if loss_history else "TODO")
print("Final mean target probability:", prob_history[-1] if prob_history else "TODO")

print("\nTarget probability at selected steps:")
for step in [0, 25, 100]:
    print(f"  step {step}: {prob_history[step]:.4f}")
print(f"  final step ({len(prob_history) - 1}): {prob_history[-1]:.4f}")

for X_example, target_index, target_name in training_examples:
    with torch.no_grad():
        Q = X_example @ learned_WQ
        K = X_example @ learned_WK
        bank_logits = (Q[0] @ K.T) / math.sqrt(d_k)
        final_probs = stable_softmax(bank_logits, dim=-1)

    print(f"\nTarget = {target_name}")
    print("Final attention distribution:", final_probs)
    if final_probs is not None:
        print("Predicted index:", int(final_probs.argmax()))
        print("Target index:", target_index)


#### What I Did

I trained `W_Q` and `W_K` with Adam for 400 steps, using the shared `bank` query's logits against each context's keys as a classification problem (cross-entropy against the correct target token). I recorded the loss and target probability at every step before the update, so step 0 shows the untrained starting point.

In [ ]:
plt.figure()
plt.plot(loss_history)
plt.xlabel("Training step")
plt.ylabel("Mean loss")
plt.title("Task 6: Training Loss for bank's Query/Key Projections")
plt.show()

plt.figure()
plt.plot(prob_history)
plt.xlabel("Training step")
plt.ylabel("Mean target attention probability")
plt.title("Task 6: Target Attention Probability During Training")
plt.show()


#### What I Did

The loss curve drops sharply and flattens near zero, and the probability curve climbs from a near-random start up toward 1.0, which shows `bank`'s query learning to prefer the correct context word in both examples at once.

**Analysis Questions**

1. Before training, what determines where `bank` attends?
2. After training, which values changed: the input embeddings, $W_Q$, $W_K$, or all of them?
3. Explain how gradients can modify an attention pattern even though there is no rule such as
   `"if bank then attend to money"`.
4. Why is it significant that the same $W_Q,W_K$ are shared across both sequences?
5. Print the target attention probability at steps 0, 25, 100, and the final step. What does
   its trajectory tell you?
6. Would training $W_V$ directly change the **attention probabilities** in this formulation?
   Why or why not?

7. Is the `bank` query different across these two contexts at a fixed training step? Explain
   why the experiment can succeed even if it is identical.
8. Why can a high mean target probability hide poorer performance on one example? Report
   each example's probability separately.


#### My Answers

1. Before training, only the random initial values of W_Q and W_K decide where bank attends — an essentially arbitrary starting pattern with no relation to meaning.

2. Only W_Q and W_K changed. They are the only tensors registered with the optimizer; the input embeddings never move, and W_V is not even used in this task's loss.

3. Gradient descent nudges W_Q and W_K a small step in whichever direction increases the score between bank's query and the correct target key relative to the others. Repeating that nudge for hundreds of steps builds up a systematic preference — no rule is ever written down, it simply emerges from many small numeric corrections that reduce the loss.

4. Sharing W_Q and W_K forces one single query-producing function to work for *both* situations at once. Since bank's embedding — and therefore its query — is identical in both contexts, the only way to succeed is for that one query to score highest against whichever candidate keys happen to be present each time. That is a much stronger form of learning than fitting two contexts separately.

5. My run printed: step 0 -> 0.2512, step 25 -> 0.6744, step 100 -> 0.9950, final step (399) -> 0.9994. The curve jumps fast early on and then keeps refining more slowly as it approaches certainty — the typical shape of gradient descent converging on an easy, well-posed target.

6. No. Attention probabilities A only depend on Q and K through softmax(QK^T); V never appears in that computation. Training W_V would change the contextualized output Z, but it would leave the attention weights A completely untouched.

7. No, the query is identical in both contexts at any fixed step, since it comes from the same bank embedding through the same W_Q. What differs is only which keys are available to compare against (money/loan/approved versus river/water/flood). The experiment works precisely because one shared query can rank differently against two different candidate sets — it never needs the query itself to change.

8. Because the mean hides how each example is doing on its own. In my run both examples happened to converge to about the same probability (money about 0.9994, river about 0.9993), so the mean is a fair summary here — but if one example were stuck at, say, 60% while the other hit 99%, the mean (around 80%) would still look fine while completely hiding that one context never learned properly. Only the per-example numbers reveal that kind of imbalance.

---
## Closing Reflection

Across this lab, bank's output vector changed whenever its surrounding words changed, and in Task 6 its query even learned to prefer the correct context word. That only shows the representation is sensitive to context: the projections mix in information from nearby tokens, and gradient descent can shape that mixing to hit a target. It does not show the model understands that "bank" can mean a financial institution or a riverbank. The numbers being moved around have no attached concept of money or water, they are just vectors and dot products being optimized to reduce a loss. Real understanding would mean the representation supports reasoning about what a bank actually is, not just that it shifts when its neighbors change or that it can be trained to rank one target above another.